# Results — the 2x2 ablation over five folds

Four loss configurations by five folds. Reads the frozen per-skull metrics in
`experiments_log/eval_all_runs.csv`, so sections 1-8 need no GPU, no weights and
no data. Section 9 optionally rebuilds one fold from its checkpoint.

How to run the notebooks: `notebooks/README.md`.

## 1 · Setup

Run names are generated from the config list, not typed, so a label cannot drift
from its directory. `fold_frame` checks the rest and raises on anything it cannot
verify.

In [ ]:
import os
import sys

REPO = os.path.abspath(".")
while REPO != os.path.dirname(REPO) and not os.path.isdir(os.path.join(REPO, "src", "models")):
    REPO = os.path.dirname(REPO)
assert os.path.isdir(os.path.join(REPO, "src", "models")), f"repo root not found from {os.getcwd()}"
for sub in ("src/models", "src/eval"):
    sys.path.insert(0, os.path.join(REPO, sub))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import importlib
import numpy as np
import pandas as pd
import report as rp
importlib.reload(rp)      # report.py changes often; without this you get the cached copy

CONFIGS = ["cd_only", "lr_fix_only", "rep_w05", "cd_rep05_full"]
N_FOLDS = 5
BASE = "cd_rep05_full"    # the cell the pooled comparison in section 8 quotes against

runs = rp.load_runs(REPO, [f"msn_skullfix/{c}_f{f}" for c in CONFIGS for f in range(N_FOLDS)])

df = pd.read_csv(os.path.join(REPO, "experiments_log", "eval_all_runs.csv"), dtype={"id": str})
df = df[df["defect_def"] == "implant"]     # the 5 mm rows are another definition, not a variant
fdf = rp.fold_frame(df[df["run"].isin([r.label for r in runs])], runs)

print(f"{len(runs)} runs, {len(CONFIGS)} configs x {N_FOLDS} folds")
print(f"{fdf['id'].nunique()} skulls, each validated exactly once per config")
print(f"architecture: {sorted({r.arch_label for r in runs})}\n")
for c in CONFIGS:
    rs = [r for r in runs if r.label.startswith(c + "_f")]
    ep = [r.meta["epochs_run"] for r in rs]
    print(f"{c:16}{rs[0].config_str():36}epochs {min(ep)}-{max(ep)}")

## 2 · What the numbers mean

**Main metric `defect_cov_mm`**: from each ground-truth point inside the defect,
the distance to the nearest predicted point. Lower is better. Only 6.18% of a
cloud is defect; the rest is surface the input already shows.

Warning: `defect_prec_mm` is gameable — predict nothing into the hole and it looks
perfect. Always read it beside `defect_n_pred`.

Every column is defined in `src/eval/report.py`; the sampling floor is
`experiments_log/sampling_floor.csv` (100 skulls, no model involved):
CD_t 4.619 +- 0.252 mm, one-way 2.310 +- 0.126 mm. About 73% of a reported
CD_t is therefore the representation, not the model. Warning: these figures
cannot be placed beside voxel-domain results, which work at 0.45 mm.

## 3 · What counts as a difference

The unit is the fold mean, and the bar is **every fold agreeing and
|delta| > 2 x SE**. No p-value: at k = 5 the smallest attainable sign-test p is
0.0625, so no rank test on folds can reach this project's p < 0.002.

Warning: one cell passes this and fails the project's separate 0.1 mm resolution
line, and both readings get reported — section 6 flags it automatically. That
line was set from an estimated SE of about 0.09, while the measured paired SEs
are 0.013-0.062, so the fold test is the more sensitive of the two.

## 4 · The 2x2

In [ ]:
METRICS = ["defect_cov_mm", "defect_HD95_mm", "defect_prec_mm", "defect_n_pred",
           "CD_t_mm", "HD95_mm", "clump_%", "spacing_CV"]
summary = rp.fold_summary(fdf, cols=METRICS)

m = summary.loc["defect_cov_mm"]
print("defect_cov_mm, mean of the five fold means, mm (lower is better)\n")
print(f"{'':12}{'no repulsion':>16}{'with repulsion':>16}")
for label, (left, right) in [("with DCD", ("lr_fix_only", "rep_w05")),
                             ("no DCD", ("cd_only", "cd_rep05_full"))]:
    print(f"{label:12}{m.loc[left, 'mean']:>16.3f}{m.loc[right, 'mean']:>16.3f}")

print("\n\nEvery metric. `mean` is the figure to report, `std_folds` the spread")
print("beside it, `se` what a difference has to beat.\n")
print(summary.to_string())

## 5 · The four edges

Each edge changes exactly one thing, paired fold by fold, so n = 5.

In [ ]:
EDGES = [("+repulsion, no DCD",     "cd_only",       "cd_rep05_full"),
         ("+DCD, no repulsion",     "cd_only",       "lr_fix_only"),
         ("+repulsion, with DCD",   "lr_fix_only",   "rep_w05"),
         ("+DCD, with repulsion",   "cd_rep05_full", "rep_w05")]

for name, base, other in EDGES:
    print(f"[{name}]")
    print(rp.format_fold_paired(fdf, base, other,
                                cols=["defect_cov_mm", "CD_t_mm", "clump_%"]))
    print()

### How to word the headline

CD + repulsion is the best cell numerically but is **indistinguishable** from
CD + DCD + repulsion: +0.016 mm, one fold of five, interval covering zero.

So not *CD + repulsion beats CD + DCD + repulsion*, but: **adding DCD on top of
repulsion has no measurable effect, so DCD is redundant and the simpler
CD + repulsion is used.** DCD remains a useful metric and is reported throughout.

Superseded by these folds, do not quote: *the two mechanisms hurt together* (they
are sub-additive, see below), *CD_t resolves none of the edges* (it resolves one),
and the single-split effect sizes.

In [ ]:
per_fold = fdf.groupby(["config", "fold"])["defect_cov_mm"].mean().unstack("config")
inter = ((per_fold["rep_w05"] - per_fold["lr_fix_only"])
         - (per_fold["cd_rep05_full"] - per_fold["cd_only"]))
se = float(inter.std(ddof=1) / np.sqrt(len(inter)))
dcd_alone = float((per_fold["lr_fix_only"] - per_fold["cd_only"]).mean())
rep_alone = float((per_fold["cd_rep05_full"] - per_fold["cd_only"]).mean())
both = float((per_fold["rep_w05"] - per_fold["cd_only"]).mean())

print(f"interaction   {inter.mean():+.3f} mm    SE {se:.3f}    t {inter.mean() / se:.2f}"
      f"    same sign {int((inter > 0).sum())}/{len(inter)}")
print(f"DCD alone {dcd_alone:+.3f},  repulsion alone {rep_alone:+.3f},"
      f"  sum {dcd_alone + rep_alone:+.3f}")
print(f"both together {both:+.3f}, which is {both / (dcd_alone + rep_alone) * 100:.0f}%"
      f" of that sum")
print("\nA positive interaction while both main effects are negative means the two")
print("overlap: together they deliver about half of what they would if they added.")

## 6 · Defect region against whole cloud

Why the evaluation protocol exists: report whole-cloud CD only, and most of these
edges disappear.

In [ ]:
def resolved(row):
    """The k-fold criterion: every fold in the same direction, and |delta| > 2 SE."""
    unanimous = row["better"] in (f"0/{N_FOLDS}", f"{N_FOLDS}/{N_FOLDS}")
    return unanimous and abs(row["delta"]) > 2 * row["fold_se"]

COMPARE = ["defect_cov_mm", "CD_t_mm"]
count = dict.fromkeys(COMPARE, 0)
under_line = []

print(f"{'edge':24}" + "".join(f"{c:>28}" for c in COMPARE))
print("-" * (24 + 28 * len(COMPARE)))
for name, base, other in EDGES:
    p = rp.fold_paired(fdf, base, other, cols=COMPARE)
    row = ""
    for c in COMPARE:
        r = p.loc[c]
        ok = resolved(r)
        count[c] += ok
        row += f"{f'{r.delta:+.3f}  {r.better}  ' + ('resolved' if ok else 'not resolved'):>28}"
        if ok and abs(r["delta"]) < 0.1:
            under_line.append((name, c, r["delta"]))
    print(f"{name:24}{row}")
print("-" * (24 + 28 * len(COMPARE)))
print(f"{'resolved':24}" + "".join(f"{str(count[c]) + f'/{len(EDGES)}':>28}" for c in COMPARE))

print("\nThe edges CD_t misses are not the marginal ones -- they are the two the")
print("defect region resolves most strongly.")
for name, c, d in under_line:
    print(f"\nWarning: '{name}' clears the fold criterion on {c} at {d:+.3f} mm but "
          f"falls under\n  the 0.1 mm resolution line (section 3). Report both readings.")

## 7 · Did anything win by training longer?

Runs stopped on their own between 200 and 360 epochs, and the reported figure is
the best epoch over the whole run. `epoch_matched` recomputes each run's
best-so-far at a common epoch. This check once removed two thirds of a claimed
0.095 mm advantage.

In [ ]:
common = min(len(r.hist) for r in runs)
em = rp.epoch_matched(runs, at=sorted({150, common}))
em["config"] = em.index.str.rsplit("_f", n=1).str[0]
print(em.groupby("config", sort=False).mean(numeric_only=True).round(3).to_string())
print(f"\nval CD_t in mm, history scale, averaged over {N_FOLDS} folds."
      f" Common epoch: {common}.")
print("Warning: this column is CD_t, not the main metric, and it is the training-time")
print("scale -- about 0.09 mm below the per-skull figure. Use it to compare positions")
print("along a curve, never as a reported value.")

### Had the main metric settled when early stopping fired?

Early stopping watches `val_loss`, conclusions are read off defect coverage. The
training callback logs `val_defect_cov_mm` every ten epochs, so the gap between
the two can be measured rather than assumed.

Warning: diagnostic only. It never selects a checkpoint — picking the best epoch
by the reported metric, on the skulls the result is reported on, would bias that
result downwards.

In [ ]:
print(f"{'config':16}{'last logged':>13}{'best logged':>13}{'gap':>9}{'tail spread':>13}")
print("-" * 64)
for c in CONFIGS:
    last, best, tail = [], [], []
    for f in range(N_FOLDS):
        s = next(r for r in runs if r.label == f"{c}_f{f}").hist["val_defect_cov_mm"].dropna()
        last.append(float(s.iloc[-1]))
        best.append(float(s.min()))
        tail.append(float(s.tail(5).std()))
    print(f"{c:16}{np.mean(last):>13.3f}{np.mean(best):>13.3f}"
          f"{np.mean(last) - np.mean(best):>+9.3f}{np.mean(tail):>13.3f}")

print("\nmm, mean over folds. The gap is what stopping on val_loss cost on the main")
print("metric: well under the 0.1 mm resolution line in every config, so the curve had")
print("flattened. These are also an independent check on section 4 -- a callback during")
print("training and report.eval_runs afterwards are separate code paths over the same")
print("masks, and they land within a few hundredths of each other, in the same order.")

## 8 · Would it hold on other skulls?

Pooling all 100 skulls: each skull is validated exactly once per config, so the
pairing is legitimate.

Warning: generalisation evidence only. Its p-value is anti-conservative for *is
this config better* — twenty skulls inside one fold share one trained model.

In [ ]:
pooled = fdf.assign(run=fdf["config"])
for c in CONFIGS:
    if c == BASE:
        continue
    print(rp.format_paired(pooled, BASE, c,
                           cols=["defect_cov_mm", "CD_t_mm", "clump_%"]))
    print()

rp.fig_per_skull(pooled, "defect_cov_mm").show()

## 9 · Recompute one fold (optional, needs a GPU)

Rebuilds one fold from its checkpoint and compares. This is the claim that
**evaluation is reproducible while training is not**. Restart the kernel before
training afterwards.

For a whole-table recompute use `src/eval/recompute_eval_all.py`, which carries
the assertion that mask-independent columns must not move.

In [ ]:
RECOMPUTE = False              # needs a GPU and the checkpoint
CHECK = f"{BASE}_f0"

if RECOMPUTE:
    got = rp.eval_runs(REPO, [r for r in runs if r.label == CHECK])
    cols = [c for c in got.columns
            if c not in ("run", "id") and pd.api.types.is_numeric_dtype(got[c])]
    was = df[df["run"] == CHECK].set_index("id")[cols].sort_index()
    now = got.set_index("id")[cols].sort_index()
    worst = float((now - was).abs().to_numpy().max())
    print(f"\n{CHECK}: largest difference against the frozen table {worst:.3e}")
    assert worst < 1e-9, "evaluation is supposed to be deterministic -- investigate"
else:
    print("RECOMPUTE = False. Set it to True on a machine holding the checkpoints.")

## 10 · Elsewhere

- Released-weights baseline, fold by fold: `MSN_baseline_pretrained.ipynb`
- Meshes and density pictures: `MSN_eval_surface.ipynb`
- One-off studies: one script each under `src/eval/`, listed in `src/eval/README.md`